## Regular water waves interacting with an array of submerged porous vertical cylinders - Frequency-Domain BEM

**3D Boundary Element Method for Linear Wave Propagation**

This tutorial presents the implementation of a 3D BEM to simulate the propagation of regular linear water waves interacting with an array of porous cylinders.

The formulation is based on:
- Linear potential flow theory  
- Frequency-domain (time-harmonic) analysis  
- Boundary integral equation formulation  

Objectives of this notebook:
- Introduce the workflow and structure of the code  
- Solve the wave structure interaction problem
- Analyse the effect of the mesh refinement on the results
- Compute wave reflection and transmission coefficients
- Compute the force components
- Iterate the porous effect parameter G (quadratic pressure drop), see Medina Rodriguez et al. (2025).


Ayrton Alfonso Medina Rodriguez, Consortium R3D, Zhenhua Huang; Wave interaction with two-dimensional submerged perforated thin-plate structures: Effects of structure shape. Physics of Fluids 1 October 2025; 37 (10): 107116. https://doi.org/10.1063/5.0279722

<p align="center">
  <img src="Sketch_08.png" width="600"><br>
  <em>Figure 1. Regular water waves interacting with an array of submerged porous vertical cylinders </em>
</p>

In [ ]:
# 1. Automatic reloading of your .py files
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(".."))    

import meshio
import numpy as np

from MAIN_MESH import MAIN_MESH

from BEM_UTILS.constants import gravity, rho

from BEM_UTILS.PROCESS_MESH import PROCESS_MESH
from BEM_UTILS.PROCESS_CONNECTIVITY import PROCESS_CONNECTIVITY
from BEM_UTILS.IDENTIFY_BOUNDARY_ELEMENTS import IDENTIFY_BOUNDARY_ELEMENTS
from BEM_UTILS.COS_DIR import COS_DIR
from BEM_UTILS.MATCHING import MATCHING

from MATRICES.GHMAT6 import GHMAT6
from MATRICES.ETAXJA import ETAXJA
from MATRICES.EXTIN6 import EXTIN6
from MATRICES.LOCIN6 import LOCIN6
from MATRICES.TRILOC6 import TRILOC6
from MATRICES.FUNDA6 import FUNDA6

from BOUNDARY_CONDITIONS.BC_FREE_SURFACE import BC_FREE_SURFACE
from BOUNDARY_CONDITIONS.BC_FAR_FIELD_IN import BC_FAR_FIELD_IN
from BOUNDARY_CONDITIONS.BC_FAR_FIELD_OUT import BC_FAR_FIELD_OUT
from BOUNDARY_CONDITIONS.BC_SCATTERING import BC_SCATTERING

from BOUNDARY_CONDITIONS.BC_FREE_SURFACE import BC_FREE_SURFACE
from BOUNDARY_CONDITIONS.BC_FAR_FIELD_IN import BC_FAR_FIELD_IN
from BOUNDARY_CONDITIONS.BC_FAR_FIELD_OUT import BC_FAR_FIELD_OUT
from BOUNDARY_CONDITIONS.BC_CONSTANT_G import BC_CONSTANT_G
from BOUNDARY_CONDITIONS.BC_SCATTERING import BC_SCATTERING
from BOUNDARY_CONDITIONS.UPDATE_G_POROUS import UPDATE_G_POROUS

from BEM_UTILS.FX_FUNCTION import FX_FUNCTION
from BEM_UTILS.ENERGY_FLUX import ENERGY_FLUX
from BEM_UTILS.NODES_FS import NODES_FS
from BEM_UTILS.POROUS_NODES import POROUS_NODES

from BEM_UTILS.FORCE import FORCE

from BEM_UTILS.VISUAL_FREE_SURFACE import VISUAL_FREE_SURFACE

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # activates 3D plotting

**2.1 Physical domain and geometry parameters**

In [ ]:
# FLUID DOMAIN
DEPTH = 2.5
WIDTH_CHANNEL = 6*DEPTH

# Structure geometry
RADIUS = 1/2*DEPTH
HEIGHT = 1/2*DEPTH

#     Point 1      2             3               4
#     ↓            ↓             ↓               ↓
#     +------------+-------------+---------------+
#     |            |   Object    |               |
#     |  Incident  |   .......   |  Transmitted  |   y
#     |     →      |   .     .   |               |   ↑
#(0,0)+.....→......|......+......|...............|    → x
#     |     →      |   .     .   |               |
#     |            |   .......   |               |
#     |            |             |               |
#     +------------+-------------+---------------+

# Point 1, 2, 3 and 4
FLUME_LENGTH_X = [0, 3*DEPTH, 5*DEPTH, 8*DEPTH]

# Location of the center's structures in x and y coordinates
# The middle of the wave flume is at y=0 
LOC_STRUCT_X = [4*DEPTH, 4*DEPTH, 4*DEPTH]
LOC_STRUCT_Y = [1.5*DEPTH, 0, -1.5*DEPTH]

# Number of structures
N_STRUCT  = len(LOC_STRUCT_X);

# Number of regions: outer and inner regions
NUM_OF_REG = 2

**2.2 Wave parameters and frequency range**

In [ ]:
# Number of frequencies / wave cases
ttmax = 10

# Dimensionless wavenumber (k*h)
kh = np.linspace(0.15, 1.00, ttmax)

# Wave steepness H/λ
HL = 0.01

# Porosity
eps = 0.10

# Porous effect parameter
Gporous_ini = 0.5

# Iterations for correction of Gporous (4 to 5 iterations are enough most of the time)
ITERATION = 4

**3. Mesh**

In [ ]:
# GMSH meshes
MAIN_MESH(DEPTH,WIDTH_CHANNEL,RADIUS,HEIGHT,FLUME_LENGTH_X,LOC_STRUCT_X,LOC_STRUCT_Y)

In [ ]:
# Read the Gmsh file and extract node and element connectivity
# Process the outer region
MESH_0_POS, MESH_0_QUADS9 = PROCESS_MESH("OUTER_REGION.msh")

# Process the inner region
MESH_1_POS, MESH_1_QUADS9 = PROCESS_MESH("INNER_REGION.msh")

In [ ]:
# Quadrilateral elements are used; each element has 9 nodes ordered counterclockwise.
NCONEC = 9;

In [ ]:
# Initialize arrays to store the number of nodes (N) and elements (NE) per region
N  = [0] * NUM_OF_REG
NE = [0] * NUM_OF_REG

In [ ]:
# Extract the number of nodes, number of elements, and reorder element connectivity
# to match the node orientation required by the BEM formulation
N[0] ,  NE[0] ,  KCONEC_0 = PROCESS_CONNECTIVITY(MESH_0_POS , MESH_0_QUADS9)
N[1] ,  NE[1] ,  KCONEC_1 = PROCESS_CONNECTIVITY(MESH_1_POS , MESH_1_QUADS9)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    MESH_0_POS[:,0],
    MESH_0_POS[:,1],
    MESH_0_POS[:,2],
    c='k',
    s=1
)

ax.scatter(
    MESH_1_POS[:,0],
    MESH_1_POS[:,1],
    MESH_1_POS[:,2],
    c='r',
    s=1
)

# DOMAIN LIMITS
x_min, x_max = np.min(MESH_0_POS[:,0]), np.max(MESH_0_POS[:,0])
y_min, y_max = np.min(MESH_0_POS[:,1]), np.max(MESH_0_POS[:,1])
z_min, z_max = np.min(MESH_0_POS[:,2]), np.max(MESH_0_POS[:,2])

Lx = x_max - x_min
Ly = y_max - y_min
Lz = z_max - z_min

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Domain")

# Aspect ratio
ax.set_box_aspect([Lx, Ly, Lz])

plt.show()

In [ ]:
# Identify elements belonging to each boundary and compute their outward normal vectors
REF_R0 , NORMAL_0 = IDENTIFY_BOUNDARY_ELEMENTS(NCONEC , KCONEC_0 , MESH_0_POS , NE[0], N_STRUCT)
REF_R1 , NORMAL_1 = IDENTIFY_BOUNDARY_ELEMENTS(NCONEC , KCONEC_1 , MESH_1_POS , NE[1], N_STRUCT)

In [ ]:
# Match central nodes (the 9th node) between adjacent regions to define the interface connectivity
# CYL 1
NODES_MATCH_CYL_1 = MATCHING(REF_R0['POROUS_31'] ,  MESH_0_POS  ,  KCONEC_0  ,  REF_R1['POROUS_31'] ,  MESH_1_POS  ,  KCONEC_1);
# CYL 2
NODES_MATCH_CYL_2 = MATCHING(REF_R0['POROUS_32'] ,  MESH_0_POS  ,  KCONEC_0  ,  REF_R1['POROUS_32'] ,  MESH_1_POS  ,  KCONEC_1);
# CYL 3
NODES_MATCH_CYL_3 = MATCHING(REF_R0['POROUS_33'] ,  MESH_0_POS  ,  KCONEC_0  ,  REF_R1['POROUS_33'] ,  MESH_1_POS  ,  KCONEC_1);

In [ ]:
# Total number of nodes and elements
N_total  = np.sum(N)
NE_total = np.sum(NE)

In [ ]:
# Concatenate local connectivity matrices and map local node indices to the global coordinate system
KCON_N_0  = KCONEC_0
KCON_N_01 = KCONEC_1 + N[0]

KCONEC = np.hstack((KCON_N_0, KCON_N_01))

**4. Influence matrices**

In [ ]:
# Form H and B matrices for each region
H0 , G0 = GHMAT6(MESH_0_POS , NCONEC , KCONEC_0 , NE[0] , N[0] , DEPTH);
H1 , G1 = GHMAT6(MESH_1_POS , NCONEC , KCONEC_1 , NE[1] , N[1] , DEPTH);

In [ ]:
# Assembling the H and G matrices
from scipy.linalg import block_diag
H = np.zeros((N_total, N_total), dtype=np.complex128)

H = block_diag(H0, H1)

G = np.zeros((N_total, NCONEC * NE_total), dtype=np.complex128)

G = block_diag(G0, G1)

In [ ]:
# Identify matching nodes across interfaces. 
# Provides a mapping of local node indices, element indices, and local ordering (1-9).
# Note: STORE_NODE_out and STORE_NODE_in maintain a 1-to-1 row correspondence.
[STORE_NODE_R0_CYL_1, STORE_NODE_R1_CYL_1] = POROUS_NODES(NCONEC, KCONEC_0, KCONEC_1, MESH_0_POS, MESH_1_POS, NODES_MATCH_CYL_1)
[STORE_NODE_R0_CYL_2, STORE_NODE_R1_CYL_2] = POROUS_NODES(NCONEC, KCONEC_0, KCONEC_1, MESH_0_POS, MESH_1_POS, NODES_MATCH_CYL_2)
[STORE_NODE_R0_CYL_3, STORE_NODE_R1_CYL_3] = POROUS_NODES(NCONEC, KCONEC_0, KCONEC_1, MESH_0_POS, MESH_1_POS, NODES_MATCH_CYL_3)

**5. Frequency loop and boundary condition application**

In [ ]:
# Preallocate arrays for reflection/transmission coefficients
CR = np.zeros(ttmax)
CT = np.zeros(ttmax)
KR = np.zeros(ttmax)
KT = np.zeros(ttmax)
KL = np.zeros(ttmax)

# Preallocate arrays for force components
FX_1_out = np.zeros(ttmax,dtype=complex)
FY_1_out = np.zeros(ttmax,dtype=complex)
FZ_1_out = np.zeros(ttmax,dtype=complex)

FX_1_in = np.zeros(ttmax,dtype=complex)
FY_1_in = np.zeros(ttmax,dtype=complex)
FZ_1_in = np.zeros(ttmax,dtype=complex)

# Dictionary to store wave properties for each frequency case
WAVE = {
    "k": np.zeros(ttmax),
    "lambda": np.zeros(ttmax),
    "HL": np.zeros(ttmax),
    "T": np.zeros(ttmax),
    "omega": np.zeros(ttmax),
    "Amp": np.zeros(ttmax)
}

Err_1 = np.zeros((ITERATION, ttmax))
Err_2 = np.zeros((ITERATION, ttmax))
Err_3 = np.zeros((ITERATION, ttmax))

# Frequency loop
for tt in range(ttmax):

    print(tt + 1)

    # Compute wavenumber from dimensionless kh
    k = kh[tt] / DEPTH

    # Linear wave dispersion relation → wave period
    T = np.sqrt((2*np.pi)**2 / (k * gravity * np.tanh(k * DEPTH)))

    # Wave frequency and angular frequency
    f = 1.0 / T
    omega = 2*np.pi*f

    # Wave height from steepness H/λ and corresponding amplitude
    Wave_height = HL * (2*np.pi / k)
    Amp = Wave_height / 2

    # Store wave properties
    WAVE["k"][tt] = k
    WAVE["lambda"][tt] = 2*np.pi / k
    WAVE["HL"][tt] = Wave_height / (2*np.pi / k)
    WAVE["T"][tt] = T
    WAVE["omega"][tt] = omega
    WAVE["Amp"][tt] = Amp

    # Porous effect parameter for all structs
    Gporous = Gporous_ini*np.ones(N_STRUCT,dtype=complex)
    G_old   = np.zeros(N_STRUCT,dtype=complex)

    # Wall thickness
    b = 0.0254 * 1.0
    l = 0.0   # b/eps if desired

    # ============================
    # NONLINEAR ITERATIONS
    # ============================

    for HHH in range(ITERATION):

        print(HHH + 1)

        # Backup original system matrices before applying BCs
        A = H.copy()
        B = G.copy()

        # ------------------------
        # Apply boundary conditions
        # ------------------------
    
        # Free-surface boundary condition
        A = BC_FREE_SURFACE(A, B, omega, gravity, NCONEC, REF_R0["FS"], KCONEC, NE, 0)
    
        # Radiation boundary conditions at inlet and outlet (far field)
        A = BC_FAR_FIELD_IN(A, B, k, NCONEC,  NORMAL_0, REF_R0["IN"],  KCONEC, NE, 0)
        A = BC_FAR_FIELD_OUT(A, B, k, NCONEC, NORMAL_0, REF_R0["OUT"], KCONEC, NE, 0)

        # Porous boundary condition
        A = BC_CONSTANT_G(A, B, k, NCONEC, STORE_NODE_R0_CYL_1, STORE_NODE_R1_CYL_1, N, NE, Gporous[0], 0, 1)
        A = BC_CONSTANT_G(A, B, k, NCONEC, STORE_NODE_R0_CYL_2, STORE_NODE_R1_CYL_2, N, NE, Gporous[1], 0, 1)
        A = BC_CONSTANT_G(A, B, k, NCONEC, STORE_NODE_R0_CYL_3, STORE_NODE_R1_CYL_3, N, NE, Gporous[2], 0, 1)        
    
        # Scattering boundary condition from incident potential at inlet
        BC_S = np.zeros_like(KCONEC[0:NCONEC, :], dtype=np.complex128)
        BC_S = BC_SCATTERING(BC_S, Wave_height, k, omega, gravity, DEPTH, MESH_0_POS, NCONEC, REF_R0["IN"], KCONEC, N, NE, 0)
    
        # Build right-hand-side vector (B · BC_S)
        FX_S = FX_FUNCTION(NCONEC, B, BC_S, N, NE)
    
        # Solve linear BEM system
        Coef_S = np.linalg.solve(A, FX_S)

        # UPDATE Gporous
        G_old[0] = Gporous[0]
        G_old[1] = Gporous[1]
        G_old[2] = Gporous[2]
        print(Gporous)
        # Gporous is corrected
        Gporous[0] = UPDATE_G_POROUS(Coef_S, k, omega, eps, l, Gporous[0], NCONEC, STORE_NODE_R0_CYL_1, STORE_NODE_R1_CYL_1, MESH_0_POS, MESH_1_POS, N, 0, 1)
        Gporous[1] = UPDATE_G_POROUS(Coef_S, k, omega, eps, l, Gporous[1], NCONEC, STORE_NODE_R0_CYL_2, STORE_NODE_R1_CYL_2, MESH_0_POS, MESH_1_POS, N, 0, 1)
        Gporous[2] = UPDATE_G_POROUS(Coef_S, k, omega, eps, l, Gporous[2], NCONEC, STORE_NODE_R0_CYL_3, STORE_NODE_R1_CYL_3, MESH_0_POS, MESH_1_POS, N, 0, 1)

        # ERROR
        Err_1[HHH, tt] = abs((Gporous[0] - G_old[0])/Gporous[0])
        Err_2[HHH, tt] = abs((Gporous[1] - G_old[1])/Gporous[1])
        Err_3[HHH, tt] = abs((Gporous[2] - G_old[2])/Gporous[2])

        # Incident wave energy flux   
        FI = (WIDTH_CHANNEL) * (1/8) * rho * gravity * (Wave_height**2) * (omega / k) * (
            0.5 * (1 + (2 * k * DEPTH) / np.sinh(2 * k * DEPTH))
         )

        # Compute reflected and transmitted energy fluxes
        FLUX_REFLE, FLUX_TRANS = ENERGY_FLUX(Coef_S, omega, rho, k, gravity, Wave_height, DEPTH, NCONEC,
            REF_R0["IN"],  MESH_0_POS,
            REF_R0["OUT"], MESH_0_POS,
            KCONEC,
            N, NE, 0, 0)

        # Reflection and transmission coefficients (amplitude-based)
        CR[tt] = np.sqrt(FLUX_REFLE / FI)
        CT[tt] = np.sqrt(FLUX_TRANS / FI)
        
        # Reflection, transmission and dissipation coefficients (energy-based)
        KR[tt] = FLUX_REFLE / FI
        KT[tt] = FLUX_TRANS / FI
        KL[tt] = 1.0 - KR[tt] - KT[tt]    

        # Force components
        FX_1_out[tt], FY_1_out[tt], FZ_1_out[tt] = FORCE(Coef_S, MESH_0_POS, NCONEC, REF_R0["POROUS_31"], KCONEC, N, NE, 0)
        FX_1_in[tt],  FY_1_in[tt],  FZ_1_in[tt]  = FORCE(Coef_S, MESH_0_POS, NCONEC, REF_R1["POROUS_31"], KCONEC, N, NE, 1)

# END LOOPS

**6. Results**

**6.1 Energy coefficients**

In [ ]:
plt.figure(15)

plt.plot(kh, CR, 'b--s', label='CR_BEM')
plt.plot(kh, CT, 'r--s', label='CT_BEM')

plt.xlim([0, 3.00])
plt.ylim([0, 1.10])

plt.xlabel('kh')
plt.ylabel('Coefficient value')

plt.legend()
plt.grid(True)

plt.savefig("CR_CT.pdf")

plt.show()

**6.2 Force components**

In [ ]:
plt.figure(100)

TERM = (rho*gravity*2*WAVE["Amp"]*RADIUS*WIDTH_CHANNEL)

CFX = np.abs(1j*rho*WAVE["omega"]*(FX_1_in + FX_1_out))/TERM
CFZ = np.abs(1j*rho*WAVE["omega"]*(FZ_1_in + FZ_1_out))/TERM

# Create figure with 2 plots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,4))

# --- Horizontal force ---
ax1.plot(kh, CFX, 'b--s', label='CFx (BEM)')
ax1.set_xlim([0,3])
ax1.set_ylim([0,1.0])
ax1.set_xlabel('kh')
ax1.set_ylabel('Force coefficient')
ax1.set_title('Horizontal force')
ax1.grid(True)
ax1.legend()

# --- Vertical force ---
ax2.plot(kh, CFZ, 'r--s', label='CFz (BEM)')
ax2.set_xlim([0,3])
ax2.set_ylim([0,1.00])
ax2.set_xlabel('kh')
ax2.set_title('Vertical force')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.savefig("FX_FZ_side_by_side.pdf")
plt.show()

**6.3 Free surface**

In [ ]:
# Free surface
PHI_R = VISUAL_FREE_SURFACE(Coef_S, k, omega, gravity, Wave_height, DEPTH, NCONEC, REF_R0["FS"], MESH_0_POS, MESH_0_POS, KCONEC_0, N, NE, 0)